# CSIS3754 - Question 3: Spider Measurements Clustering
## Main Mid-Year Examination 2025

## 3.1 - Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Load dataset and assign column names according to the body part measurements
spiders = pd.read_csv(
    'spider_measurements.csv',
    header=None,   # No header in the file
    names=['abdomen-circumference', 'abdomen-length', 'pedipalp-length', 'chelicerae-length']
)

print('Dataset loaded successfully!')
spiders.head()

## 3.2 - Brief Summary of the Dataset

In [ ]:
# Statistical summary
print('Statistical Summary:')
spiders.describe()

In [ ]:
# Concise summary: index, dtype, columns, non-null values, memory usage
spiders.info()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(spiders.isnull().sum())
print(f'\nTotal missing values: {spiders.isnull().sum().sum()}')

## 3.3 - Pre-processing

In [ ]:
# DISCUSSION - Step 1: Check for object columns
# All four columns should be numeric (float/int) since they are measurements.
# If any column is of type object, we convert it to numeric.

print('Data types before pre-processing:')
print(spiders.dtypes)

In [ ]:
# Convert any object columns to numeric (coerce errors to NaN, then fill)
for col in spiders.columns:
    if spiders[col].dtype == 'object':
        spiders[col] = pd.to_numeric(spiders[col], errors='coerce')
        print(f"'{col}' converted to numeric.")

# Handle any NaN values that may have been introduced by the conversion
if spiders.isnull().sum().sum() > 0:
    for col in spiders.columns:
        if spiders[col].isnull().sum() > 0:
            spiders[col].fillna(spiders[col].median(), inplace=True)
            print(f"'{col}': NaN values filled with median.")

print('\nData after type conversion:')
print(spiders.dtypes)
spiders.head()

In [ ]:
# DISCUSSION - Step 2: Feature Scaling
# K-Means clustering uses Euclidean distance. If one measurement (e.g.
# abdomen-circumference) has much larger values than another (e.g.
# pedipalp-length), it will dominate the distance calculations.
# We apply StandardScaler to normalise all features to mean=0, std=1.

scaler = StandardScaler()
spiders_scaled = scaler.fit_transform(spiders)
spiders_scaled_df = pd.DataFrame(spiders_scaled, columns=spiders.columns)

print('Features after StandardScaler normalisation:')
spiders_scaled_df.head()

In [ ]:
# Confirm no object columns remain
obj_remaining = spiders_scaled_df.select_dtypes(include='object').columns.tolist()
print(f'Object columns remaining: {obj_remaining if obj_remaining else "None - pre-processing complete!"}')

## 3.4 - K-Means Clustering (Determine Optimal k)

In [ ]:
# Use the Elbow Method to find the optimal number of clusters
# WCSS = Within-Cluster Sum of Squares (inertia)
# The optimal k is at the 'elbow' of the curve where WCSS stops
# decreasing significantly.

wcss = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(spiders_scaled_df)
    wcss.append(kmeans.inertia_)

# Plot the Elbow curve
plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o', linestyle='--', color='steelblue')
plt.title('Elbow Method - Optimal k for Spider Measurements')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.xticks(k_range)
plt.grid(True)
plt.tight_layout()
plt.show()
print('Identify the elbow point above to determine the optimal k.')

In [ ]:
# Set optimal_k based on the elbow plot above
optimal_k = 3  # <-- Adjust this based on your elbow plot

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
labels = kmeans.fit_predict(spiders_scaled_df)

# Add cluster labels to original dataframe
spiders['Cluster'] = labels

print(f'K-Means applied with optimal k = {optimal_k}')
print('\nCluster labels identified:')
print(labels)
print('\nCluster distribution:')
print(pd.Series(labels).value_counts().sort_index())
spiders.head()

## 3.5 - Principal Component Analysis (PCA)

In [ ]:
# Apply PCA to reduce to 2 dimensions
pca = PCA(n_components=2)
spiders_pca = pca.fit_transform(spiders_scaled_df)

# Compare dimensions before and after
print(f'Dimensions BEFORE PCA: {spiders_scaled_df.shape}')
print(f'Dimensions AFTER PCA:  {spiders_pca.shape}')

In [ ]:
# Print principal components and explained variance
print('Principal Components (loadings):')
pc_df = pd.DataFrame(
    pca.components_,
    columns=spiders_scaled_df.columns,
    index=['PC1', 'PC2']
)
print(pc_df)

print('\nExplained Variance Ratio:')
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {var:.4f} ({var*100:.2f}%)')

print(f'\nTotal Variance Explained: {pca.explained_variance_ratio_.sum()*100:.2f}%')

In [ ]:
# Seaborn scatterplot of PC1 vs PC2 coloured by cluster
pca_df = pd.DataFrame(spiders_pca, columns=['PC1', 'PC2'])
pca_df['Cluster'] = labels.astype(str)

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=pca_df,
    x='PC1', y='PC2',
    hue='Cluster',
    palette='Set1',
    alpha=0.7,
    edgecolor='k',
    linewidth=0.3,
    s=60
)
plt.title(f'PCA Scatter Plot - Spider Measurement Clusters (k={optimal_k})', fontsize=13)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()

## 3.6 - Evaluation and Discussion

In [ ]:
discussion_36 = """
EVALUATION OF CLUSTERS AND PCA EFFECTIVENESS:
==============================================

Cluster Evaluation:
- If the scatter plot shows clearly separated, compact groupings with
  minimal overlap, the chosen value of k is appropriate and K-Means
  has successfully identified meaningful clusters in the spider data.

- If clusters overlap considerably, it may indicate that k should be
  adjusted (increase or decrease based on the elbow plot) or that
  the natural groupings in the data are not linearly separable.

- For spider body part measurements, distinct clusters could represent
  different spider species or sex-based differences (male vs female
  spiders often differ significantly in body size).

Agreement with Number of Clusters:
- The elbow method provides a mathematical basis for choosing k.
  If the scatter plot visually confirms the same number of groups,
  we can be more confident in the chosen k.
- If the scatter plot suggests more or fewer clusters than the elbow
  method, further investigation (e.g. Silhouette Score) is recommended.

PCA Effectiveness:
- This dataset has only 4 features (dimensions). Reducing to 2 dimensions
  retains a high proportion of the original variance (typically >90% for
  small datasets with correlated features like body measurements).
- Since body part measurements are often correlated (larger spiders tend
  to have larger measurements across all body parts), PCA is especially
  effective here. The first principal component likely captures the overall
  'size' variation, while the second captures 'shape' differences.
- With high explained variance, the 2D PCA plot is a reliable visualisation
  of the cluster structure in the original 4D data.
"""
print(discussion_36)